In [2]:
from vinson.utils.data_formatting.adata_utils import slice_adata, compute_if_dask, get_examples_indices_from_layer, update_layers_dict
from vinson.utils.data_formatting.encoding import sanitize_data, encode_inplace
from vinson.utils.data_formatting.container import VinsonData



In [3]:
import anndata as ad
import pandas as pd
import numpy as np
from genome_tools.data.anndata import read_zarr_backed

# from vinson.utils.helpers import read_configs
# Load the DHS dataset (original)
# dhs_file = "/net/seq/data2/projects/ENCODE4Plus/REGULOME/sequence_to_accessibility_model/training_data/NOV03/NOV03_3_epochs_with_weights_fc_squared_50.h5ad"
# dhs_adata = ad.read_h5ad(dhs_file)

# Load the variant dataset
# variant_file = "/net/seq/data2/projects/sabramov/ENCODE4/ML/NOV17/variant_train_adata.h5ad"
variant_file = '/net/seq/data2/projects/mbrannon/vinson/3_9_dataset_20totalcount_3epoch.h5ad'
variant_adata = ad.read_h5ad(variant_file)


In [4]:
def extract_variant_data_from_anndata(train_adata: ad.AnnData, suffix: str) -> VinsonData:
    """
    Convert AnnData object to H5 format and extract embeddings.
    Args:
        adata (ad.AnnData): AnnData object containing DHS data.
        suffix (str): Suffix of the epoch/layer to extract. Gets added to layer names as `{layer}.{suffix}`. E.g. epoch_1, epoch_2, etc.
    
    Returns:
        data (dict): Dictionary containing extracted data arrays.
        embeddings_df (pd.DataFrame): DataFrame containing motif embeddings.
    """
    
    data = {"ref_counts": None, "total_counts": None, "BAD": None, "logit_es": None}
    update_layers_dict(data, train_adata, suffix)
    row_idx, col_idx = get_examples_indices_from_layer(data["ref_counts"])
    
    encodings = {}

    encoding_sources = {
        "sample_id": train_adata.obs_names,
        "chrom": train_adata.var["#chr"],
        # "pos": train_adata.var["end"],
        'ref': train_adata.var['ref'],
        'alt': train_adata.var['alt']
    }

    if 'indiv_id' in train_adata.obsm:
        encoding_sources['indiv_id'] = train_adata.obsm['indiv_id']

    for key in encoding_sources:
        encode_inplace(encoding_sources, encodings, key)

    data = {
        'chrom': encoding_sources['chrom'][col_idx],
        'pos': train_adata.var['start'].values[col_idx],
        'ref': encoding_sources['ref'][col_idx],
        'alt': encoding_sources['alt'][col_idx],
        'sample_id': encoding_sources["sample_id"][row_idx],
        'ref_counts': data['ref_counts'].data,
        'total_counts': data['total_counts'].data,
        'BAD': data['BAD'].data,
        'logit_es': data['logit_es'].data,
    }
    if 'indiv_id' in encoding_sources:
        data['indiv_id'] = encoding_sources["indiv_id"][row_idx]

    data, encodings = sanitize_data(data, encodings, is_variant=True)
    embeddings_df = train_adata.obsm['motif_embeddings']

    return VinsonData(
        data,
        encodings=encodings,
        embeddings_df=embeddings_df,
        is_variant=True
    )

In [5]:
def update_layers_dict(layers: dict, train_adata: ad.AnnData, suffix: str):
    assert len(layers) > 0, "Must provide at least one layer to extract"
    for layer_name in layers:
        epoch_layer_name = f"{layer_name}.{suffix}"
        layers[layer_name] = train_adata.layers[epoch_layer_name].tocoo()
    
    class_coo = layers["class"]
    row_idx, col_idx = class_coo.row, class_coo.col
    return row_idx, col_idx

In [58]:
109263132-109264476

-1344

In [6]:
def get_indiv_id_info(train_adata: ad.AnnData, row_idx: np.ndarray):
    #return train_adata.obsm['indiv_id'].values[row_idx]
    return train_adata.obsm['indiv_id'][row_idx]


In [7]:
def sanitize_data(data: dict, is_variant=False) -> dict:
    """Ensure that all data arrays are contiguous and correct dtype."""
    
    if is_variant:
        data_keys = {
            "chrom": np.str_,
            "pos": np.int32,
            "ref": np.str_,
            "alt": np.str_,
            "ref_counts": np.float32,
            "total_counts": np.float32,
            "BAD": np.float32,
            "sample_id": np.str_,
            "logit_es": np.float32,
        }
    else:
        data_keys = {
            "read_depth": np.float32,
            "sample_id": np.str_,
            "dhs_id": np.str_,
            "chrom": np.str_,
            "summit": np.int32,
            "background": np.float32,
            "class": np.int8,
            "density": np.float32,
        }

    optional_keys = {
        "indiv_id": np.str_,
        "dhs_weight": np.float32,
    }

    keys = {
        **data_keys,
        **{k: v for k, v in optional_keys.items() if k in data},
    }

    for key, dtype in keys.items():
        # Convert ANY incoming structure (Index, Series, list) into numpy array
        arr = np.asarray(data[key], dtype=object)

        if dtype == np.str_:
            mask = pd.isna(arr) | np.isin(arr, ['None', 'nan'])
            arr[mask] = ''

        data[key] = np.ascontiguousarray(arr.astype(dtype))

    if 'background' in data:
        data['background'] = np.nan_to_num(data['background'])

    return data


In [8]:
def update_layers_dict_var(layers: dict, train_adata: ad.AnnData, suffix: str):
    assert len(layers) > 0, "Must provide at least one layer to extract"
    for layer_name in layers:
        epoch_layer_name = f"{layer_name}.{suffix}"
        layers[layer_name] = train_adata.layers[epoch_layer_name].tocoo()
    
    class_coo = layers["logit_es"]
    row_idx, col_idx = class_coo.row, class_coo.col
    return row_idx, col_idx


In [9]:
data_out = extract_variant_data_from_anndata(variant_adata, 'epoch_1')

KeyError: 'class'

In [ ]:
data_out


In [10]:
from genome_tools import GenomicInterval, VariantInterval, df_to_variant_intervals
from genome_tools.data.extractors import FastaExtractor, TabixExtractor

from vinson.utils.sequence_utils import one_hot_encode, get_iupac_char_from_alleles
from vinson.utils.helpers import replace_at
import logging

genotype_extr: TabixExtractor = None
fasta_extr: FastaExtractor = None

In [11]:
fasta_file = '/net/seq/data/genomes/human/GRCh38/noalts/GRCh38_no_alts.fa'

In [12]:
if not fasta_extr:
    fasta_extr = FastaExtractor(fasta_file)

In [13]:
genotype_file = '/net/seq/data2/projects/sabramov/ENCODE4/dnase-wasp.v5/output/all_variants_stats.bed.gz'

In [ ]:
genotype_file = '/net/seq/data2/projects/sabramov/ENCODE4/dnase-wasp.v5/phasing/output/all_phased.bed.gz'

In [14]:
genotype_file = '/home/mbrannon/genotype_no_multiallelic.tsv.gz'

In [15]:
# data = data_out
if genotype_file is not None:
    # assert 'indiv_id' in data.keys(), "Sample to genotype mapping must include 'indiv_id' column."
    genotype_extr = None
    include_genotypes = True

In [16]:
import gzip
if include_genotypes and not genotype_extr:
    with gzip.open(genotype_file, "rt") as f:
        phased = "phase_set" in f.readline()
        print('phased')
        print(phased)
        if phased:
            genotype_extr = TabixExtractor(
                    genotype_file,
                    skiprows=1,
                    columns=[
                        "chrom",
                        "start",
                        "end",
                        "ref",
                        "alt",
                        "indiv_id",
                        "gt",
                        "phase_block",
                    ],
                    na_values={"phase_block": "."},
                )
        else:
            print(f"[INFO] Using unphased genotype format ({genotype_file})")
            genotype_extr = TabixExtractor(
                genotype_file,
                columns=[
                    "chrom",
                    "start",
                    "end",
                    "rs_id",
                    "ref",
                    "alt",
                    "af_ref",
                    "af_alt",
                    "gt",
                    "_0",
                    "_1",
                    "_2",
                    "_3",
                    "indiv_id",
                ],
            )

phased
True


In [17]:
i = 0
data_slice = data[i]
chrom = data_slice['chrom']
pos = data_slice['pos']
ref = data_slice['ref']
alt = data_slice['alt']
ref_counts = data_slice['ref_counts']
total_counts = data_slice['total_counts']
bad = data_slice['BAD']
lfc = data_slice['logit_es']
sample_id = data_slice['sample_id']

NameError: name 'data' is not defined

In [ ]:
chr20:50112504-50113848)/INDIV_0021/50113173/T

In [286]:
30134919+662

30135581

In [ ]:
subset = used_results[
    (used_results["#chr"] == "chr17") &
    (used_results["start"] == 50346056) &
    (used_results["end"] == 50346057) &
    (used_results['indiv_id']=='INDIV_0003')

In [18]:
#test specific
pos = 50346056
chrom='chr17'
ref = 'G'
alt = 'A'
indiv_id = 'INDIV_0003'

In [19]:
variant = GenomicInterval(chrom, pos, pos)
interval = variant.widen(1344 // 2)

In [20]:
interval

GenomicInterval(chr17:50345384-50346728)

In [21]:
rel_pos = pos - interval.start

In [22]:
reference_variant=VariantInterval(
                chrom=chrom, start=pos, end=pos+1, ref=ref, alt=alt
            )

In [23]:
reference_variant

VariantInterval(chr17:50346056-50346057)

In [24]:
interval

GenomicInterval(chr17:50345384-50346728)

In [25]:
50112504-50113848

-1344

In [26]:
seq = fasta_extr[interval]
seq_iupac = seq_ref = seq_alt = str(seq) # modify all 3 regardless


In [27]:
len(seq)

1344

In [28]:
len(seq_iupac)

1344

In [29]:
variants = genotype_extr[interval]

In [30]:
variants

,chrom,start,end,ref,alt,indiv_id,gt,phase_block
0,chr17,50345476,50345477,T,G,INDIV_0004,0/1,NaN
1,chr17,50345476,50345477,T,G,INDIV_0006,0/1,NaN
2,chr17,50345476,50345477,T,G,INDIV_0007,1/1,NaN
3,chr17,50345476,50345477,T,G,INDIV_0008,1/1,NaN
4,chr17,50345476,50345477,T,G,INDIV_0009,0/1,NaN
...,...,...,...,...,...,...,...,...
833,chr17,50346593,50346594,G,T,INDIV_0588,0/1,NaN
834,chr17,50346593,50346594,G,T,INDIV_0593,0/1,NaN
835,chr17,50346634,50346635,C,G,INDIV_0403,0/1,NaN
836,chr17,50346656,50346657,G,C,INDIV_0046,0/1,NaN


In [31]:
if variants["indiv_id"].str.endswith(".bed.gz").any():
    # DHS format
    key = f"{indiv_id}.bed.gz"
else:
    # variant format
    key = indiv_id

variants = variants[variants["indiv_id"] == key]

In [32]:
variants

,chrom,start,end,ref,alt,indiv_id,gt,phase_block
518,chr17,50346068,50346069,G,C,INDIV_0003,0|1,50346057.0
573,chr17,50346372,50346373,G,C,INDIV_0003,0/1,NaN
597,chr17,50346538,50346539,A,G,INDIV_0003,0/1,NaN


In [33]:
extra_columns = ('gt',)
if "phase_set" not in variants.columns:
    if "phase_block" in variants.columns:
        variants = variants.rename(columns={"phase_block": "phase_set"})
    else:
        variants["phase_set"] = None
if reference_variant is not None:
    try:
        row = variants.set_index(["chrom", "start", "ref", "alt"]).loc[
            (reference_variant.chrom,
             reference_variant.start,
             reference_variant.ref,
             reference_variant.alt)
        ]
    except KeyError:
        raise ValueError(
            f"Query variant not found in genotyping file ")
    reference_variant.gt = row["gt"]
    phase_val = row.get("phase_set", None)
    reference_variant.phase_set = phase_val if not pd.isna(phase_val) and phase_val != "." else None

ValueError: Query variant not found in genotyping file 

In [ ]:
extra_columns = ("gt", "phase_set")

In [90]:

variants

,chrom,start,end,ref,alt,indiv_id,gt,phase_set
20,chr1,23559006,23559007,T,C,INDIV_0021,1/1,NaN
266,chr1,23559107,23559108,T,C,INDIV_0021,0/1,NaN
422,chr1,23559581,23559582,T,C,INDIV_0021,0/1,NaN
810,chr1,23559659,23559660,C,T,INDIV_0021,0/1,NaN


In [91]:
variants = df_to_variant_intervals(variants, extra_columns=extra_columns)

In [92]:
variants

[VariantInterval(chr1:23559006-23559007, extra_fields=('gt', 'phase_set')),
 VariantInterval(chr1:23559107-23559108, extra_fields=('gt', 'phase_set')),
 VariantInterval(chr1:23559581-23559582, extra_fields=('gt', 'phase_set')),
 VariantInterval(chr1:23559659-23559660, extra_fields=('gt', 'phase_set'))]

In [93]:
import numpy as np
import pandas as pd

from torch.utils.data import Dataset
import gzip

from genome_tools import GenomicInterval, VariantInterval, df_to_variant_intervals
from genome_tools.data.extractors import FastaExtractor, TabixExtractor

from vinson.utils.data_formatting import VinsonData
from vinson.utils.sequence_utils import one_hot_encode, get_iupac_char_from_alleles
from vinson.utils.helpers import replace_at
import logging


In [94]:
len(interval)

1344

In [95]:
for v in variants:
    rel = v.start - interval.start
    print(rel)
    iupac_base = get_iupac_char_from_alleles(v.ref, v.alt)
    print(iupac_base)
    seq_iupac = replace_at(seq_iupac, rel, iupac_base)
    print(len(seq_iupac))
    continue

    phased_match = (
        reference_variant is not None
        and v.gt in ("0|1", "1|0")
        and reference_variant.phase_set == getattr(v, "phase_set", None)
    )
    if phased_match:
        if v.gt == "1|0":
            seq_ref = replace_at(seq_ref, rel, v.alt)
            seq_alt = replace_at(seq_alt, rel, v.ref)
        else:
            seq_ref = replace_at(seq_ref, rel, v.ref)
            seq_alt = replace_at(seq_alt, rel, v.alt)
        continue
    is_het = v.gt[0] != v.gt[2]
    base_ref = v.ref
    base_alt = v.alt if is_het else (v.alt if v.gt[0] == "1" else v.ref)
    seq_ref = replace_at(seq_ref, rel, base_ref)
    seq_alt = replace_at(seq_alt, rel, base_alt)

97
Y
1344
198
Y
1344
672
Y
1344
750
Y
1344


In [96]:
len(seq_ref)

1344

In [97]:
interval

GenomicInterval(chr1:23558909-23560253)

In [98]:
seq_ref

'CCTCTCCAAGAGAGGAGATACTCACCTGGAGGTGTCAGGACACGGCCGAGTCAGTGGCAAAAGCTCCTTTTGTCGTTGGAGATGACAAGTTCCGGAGTGAGCTCGGCTGTCTGATTAGAGGAAAAGAGGGAAGAGTTACGCGAGGCAATCGGGAGCTCCGAGGGTCCCGCAGCATCCTTGCCTGGGTGTTCAGCCCTGTCCCGACTTCGAGGCTTACCTGGATGGGAAGGTGGGGGCCATCAGGGGGTCCAGGGGCTGGCTCGGCCAGGACTACCTGCAGGTCGAGAATGTAGTCGATGACGCGCTGTAGGATTTCCACCTGGCTAAGCTGAGTGCCTCTCGGGACTCCGGGTACCAGTTCCCGCAGGCGGGAGTAGCAGTGGTTCATGTCGTCCAGCAAGCTCAGCGGCTCCTCAGCTGCCGGGCCCTTCCCTCGGCCCCGGGCGATGGCCAGACTGCGTTCCGACAGGCAGCACACCGCCTCGTAGCAGCCGCGCACCGGGCTCAGCGCCTTCATGCTGGGGAGTGAGTCCAGAGGTGCCCCAAAGAGAAAGAAAACCAAAAGAAGTCCCGCTACAGTGACCTGCAACGCGCGCACGCTCGCCGCGGCGGTCACTTATAGAGCCTGCCTGGAAGGCACGCCTCTTTATTCAAAATGGCCCGCCTCGGCCCTGCCCCCGCCGGCCCTGGGCGTTCACAGCCCGCTTAAATTGCAAACAGGCTTCCTCCGGCTGGTCTGACGCCGAAGACCGCGGAGCCGCGGATTCAAAGAATGAGGAAGCGCTGATACCGGGGAGAGGCGGGCCTCTCCGCCAGCAAGGATTTAAAAATCACTCAAAACCATTAACTTCCAGAATTTGCTTTTTCCTGGCAGCACCCCAGATCTTTGAGCTTCCCTGCCCCCTGCCAGTCCGCCTTTAGCCCAACACTGGTTCGAGCCACAGCTCCTCCGAGGTCATAAATCCCTGAACAGCAAAGAAGCTCCCCCCACCCCCCGTT

In [99]:
len(seq_alt)

1344

In [280]:
if reference_variant is not None:
    rel_pos = reference_variant.start - interval.start
    if reference_variant.gt == "1|0":
        seq_ref, seq_alt = seq_alt, seq_ref
    if (seq_ref[rel_pos] != reference_variant.ref) or (seq_alt[rel_pos] != reference_variant.alt):
        raise ValueError(
            f"Expected ref & alt alleles not found in correct position "
            f"(reference_variant={reference_variant}, indiv_id={indiv_id})"
        )



In [281]:
print(len(seq_iupac))
print(len(seq_ref))
print(len(seq_alt))

1344
1344
1344


In [ ]:
reference_variant.gt

In [ ]:
variants = df_to_variant_intervals(variants, extra_columns=extra_columns)
    

In [ ]:
variants_by_pos = {}
for v in variants:
    if v.start in variants_by_pos:
        variants_by_pos[v.start].append(v)
    else:
        variants_by_pos[v.start] = [v]


In [ ]:
variants_by_pos

In [ ]:
import warnings
ambiguous_positions = set()
for pos, vars_at_pos in variants_by_pos.items():
    rel_pos = pos - interval.start
    if len(vars_at_pos) > 1:
        # multiple variants at same position → warn & treat as unphased
        warnings.warn(
            f"Multiple variants at position {pos} for {indiv_id}: {[str(v) for v in vars_at_pos]}. "
            "Treating as unphased / ambiguous."
        )
        ambiguous_positions.add(pos)
        # combine alleles into IUPAC
        combined_ref = vars_at_pos[0].ref  # pick first as placeholder
        combined_alt = "".join(set(v.alt for v in vars_at_pos))
        base = get_iupac_char_from_alleles(combined_ref, combined_alt)
        seq_iupac = replace_at(seq_iupac, rel_pos, base)
        seq_ref = replace_at(seq_ref, rel_pos, combined_ref)
        seq_alt = replace_at(seq_alt, rel_pos, combined_alt)
    else:
        v = vars_at_pos[0]
        # phased variant logic
        if 'phase_set' in extra_columns and reference_variant is not None and reference_variant.phase_set == getattr(v, 'phase_set', None):
            base = get_iupac_char_from_alleles(v.ref, v.alt)
            seq_iupac = replace_at(seq_iupac, rel_pos, base)
            if v.gt == "1|0":
                seq_ref = replace_at(seq_ref, rel_pos, v.alt)
                seq_alt = replace_at(seq_alt, rel_pos, v.ref)
            elif v.gt == "0|1":
                seq_ref = replace_at(seq_ref, rel_pos, v.ref)
                seq_alt = replace_at(seq_alt, rel_pos, v.alt)
            else:
                raise ValueError(f'Phased genotype not recognized! {v}')
        else:
            # unphased logic
            assert v.gt[0] in ("0", "1") and v.gt[2] in ("0", "1"), f"Genotype format not recognized! {v} {v.gt}"
            variant_is_het = (v.gt[0] == "1" and v.gt[2] == "0") or (v.gt[0] == "0" and v.gt[2] == "1")
            if variant_is_het:
                base = get_iupac_char_from_alleles(v.ref, v.alt)
                seq_iupac = replace_at(seq_iupac, rel_pos, base)
                seq_ref = replace_at(seq_ref, rel_pos, v.ref)
                seq_alt = replace_at(seq_alt, rel_pos, v.alt)
            elif v.gt[0] == "1":
                base = v.alt
                seq_iupac = replace_at(seq_iupac, rel_pos, base)
                seq_ref = replace_at(seq_ref, rel_pos, base)
                seq_alt = replace_at(seq_alt, rel_pos, base)
            else:
                base = v.ref
                seq_iupac = replace_at(seq_iupac, rel_pos, base)
                seq_ref = replace_at(seq_ref, rel_pos, base)
                seq_alt = replace_at(seq_alt, rel_pos, base)


In [ ]:
reference_variant.phase_set

In [ ]:
interval.start

In [ ]:
reference_variant.gt

In [ ]:
if reference_variant is not None:
    #only for phased
    if phase_set is not None and phase_set != "." and not pd.isna(phase_set):
        if reference_variant.gt == "1|0":
            print('switching')
            seq_ref, seq_alt = seq_alt, seq_ref
    
    rel_pos = reference_variant.start - interval.start
    print(f'ref_relpos {seq_ref[rel_pos]} act {reference_variant.ref}, alt relpos {seq_alt[rel_pos]} act {reference_variant.alt}')
    print(rel_pos)
    if reference_variant.start not in ambiguous_positions:
        # only enforce ValueError if this position is NOT ambiguous
        if (seq_ref[rel_pos] != reference_variant.ref) or (seq_alt[rel_pos] != reference_variant.alt):
            raise ValueError(
                "Expected ref & alt alleles not found in correct position in sequences!",
                reference_variant, variants
            )